# Multi-task design ablation (masked) --- paper Fig. 5

Reruns the four alpha-routing variants on the **round-6 masked** CSGNNv2, matched extended
(150-epoch) budget, 3 seeds, grouped split --- so Fig. 5 and the routed-vs-coupled
R^2 / vulnerability-AUC sentence reflect the deployed masked model.

Variants: **specialist** (risk-only, no aux loss), **routed** (alpha=0, ours),
**coupled** (alpha=1, aux gradients into the trunk), **uncertainty** (alpha=1 + homoscedastic
uncertainty weighting). Runtime -> GPU, then Run all. Reads everything from
`smart_load_shield_boost` on Drive (no upload).

In [ ]:
from google.colab import drive
import glob, os, sys
drive.mount('/content/drive')
c = glob.glob('/content/drive/MyDrive/**/smart_load_shield_boost', recursive=True)
FOLDER = c[0] if c else '/content/drive/MyDrive/smart_load_shield_boost'
sys.path.insert(0, FOLDER)
need = ['contingency_data.npz', 'grouped_split.npz', 'boost_core.py']
missing = [f for f in need if not os.path.exists(os.path.join(FOLDER, f))]
assert not missing, 'missing in ' + FOLDER + ': ' + str(missing)
assert 'base_mask' in open(os.path.join(FOLDER, 'boost_core.py')).read(), \
    'boost_core.py in FOLDER is the OLD unmasked version -- replace with round-6 masked'
print('FOLDER =', FOLDER, '| masked boost_core OK')

In [ ]:
import numpy as np, torch, json
import boost_core as B
from sklearn.metrics import roc_auc_score
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', DEV)
d = B.build_data(os.path.join(FOLDER, 'contingency_data.npz'),
                 os.path.join(FOLDER, 'grouped_split.npz'), DEV)
SEEDS = [0, 1, 2]
EPOCHS = 150      # extended budget: every variant compared near its ceiling (paper wording)

# matched-budget configs (all share the masked CSGNNv2 via train_eval -> set_base_mask)
CONFIGS = {
    'specialist':  dict(alpha=0.0, branched=False, zero_aux=True),   # risk-only reference
    'routed':      dict(alpha=0.0, branched=False),                  # ours (alpha=0 routing)
    'coupled':     dict(alpha=1.0, branched=False),                  # aux gradients into trunk
    'uncertainty': dict(alpha=1.0, branched=False, uncert=True),     # homoscedastic weighting
}

@torch.no_grad()
def aux_metrics(model, dd, idx):
    # identity V_min R^2 (converged) + per-bus vulnerability ROC-AUC (converged)
    model.eval(); VM, VU = [], []
    for s in range(0, len(idx), 4096):
        b = idx[s:s+4096]; _, vm, vu, _, _ = model(dd['NF'][b], dd['DENSE'][b], dd['ADJ'][b])
        VM.append(vm.float()); VU.append(torch.sigmoid(vu).float())
    VM = torch.cat(VM); VU = torch.cat(VU)
    conv = (dd['Td'][idx] == 0)
    vt = dd['Tv'][idx][conv].cpu().numpy(); vp = VM[conv].cpu().numpy()
    r2 = float(1 - ((vt-vp)**2).sum() / ((vt-vt.mean())**2).sum())
    ut = dd['Tu'][idx][conv].cpu().numpy().ravel(); up = VU[conv].cpu().numpy().ravel()
    auc = float(roc_auc_score(ut, up))
    return r2, auc

res = {}
for name, kw in CONFIGS.items():
    accs, r2s, aucs = [], [], []
    for sd in SEEDS:
        te, model = B.train_eval(d, seed=sd, epochs=EPOCHS, amp=(DEV=='cuda'), **kw)
        accs.append(te['acc'])
        if name in ('routed', 'coupled'):          # only these two feed the R^2/AUC sentence
            r2, auc = aux_metrics(model, d, d['idx_te']); r2s.append(r2); aucs.append(auc)
        print('  %-11s seed%d  acc %.2f%%  fs %.3f%%%s'
              % (name, sd, te['acc']*100, te['false_safe']*100,
                 ('  R2 %.3f AUC %.3f' % (r2s[-1], aucs[-1])) if r2s else ''))
    res[name] = dict(acc_mean=float(np.mean(accs)), acc_std=float(np.std(accs)), acc_seeds=accs)
    if r2s: res[name].update(vmin_r2=float(np.mean(r2s)), vuln_auc=float(np.mean(aucs)))
    print('%-11s MEAN acc %.2f+-%.2f%%\n' % (name, np.mean(accs)*100, np.std(accs)*100))

json.dump(res, open(os.path.join(FOLDER, 'ablation_boost_results.json'), 'w'), indent=2)
print('saved ablation_boost_results.json to', FOLDER)
print('\n=== paste into make_boost_ablation_fig.py VARIANTS (mean acc %) ===')
for k in ('specialist', 'routed', 'coupled', 'uncertainty'):
    print('  %-11s %.2f' % (k, res[k]['acc_mean']*100))
print('routed:  R2 %.3f  AUC %.3f' % (res['routed'].get('vmin_r2', float('nan')), res['routed'].get('vuln_auc', float('nan'))))
print('coupled: R2 %.3f  AUC %.3f' % (res['coupled'].get('vmin_r2', float('nan')), res['coupled'].get('vuln_auc', float('nan'))))
print('coupling costs %.2f pp, uncertainty costs %.2f pp vs routed'
      % ((res['routed']['acc_mean']-res['coupled']['acc_mean'])*100,
         (res['routed']['acc_mean']-res['uncertainty']['acc_mean'])*100))